In [122]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, GenerationConfig
import random

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [3]:
MODEL_ID = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype="auto").to(device)

Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 7698.95it/s]


In [132]:
def shuffle_image_dict(image_paths):
    shuffled_image_dict = []
    for idx, img in enumerate(image_paths):
        shuffled_image_dict.append({'original_idx' : idx, 'img_path' : img})

    random.shuffle(shuffled_image_dict)
    return shuffled_image_dict

def construct_comparison_prompt(shuffled_image_dict):
    messages = []
    messages.append({"role": "system", "content": "Your job is to decide which image you prefer."})
    comparison_prompt_content = []
    for idx, img_path in enumerate(shuffled_image_dict):
        comparison_prompt_content.append({"type": "text", "text": "Image " + str(idx + 1) + ":"})
        comparison_prompt_content.append({"type": "image", "path": img_path['img_path']})
    comparison_prompt_content.append({"type": "text", "text": "Which image do you prefer most? Only say the number of the image."})
    messages.append({"role": "user", "content": comparison_prompt_content})
    return messages

def prepare_inference(prompt, processor):
    inputs = processor.apply_chat_template(
        prompt,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=True,
    ).to(device)
    input_len = inputs["input_ids"].shape[-1]
    print(input_len)
    generation_config = GenerationConfig(max_new_tokens=1024, early_stopping=True)
    return inputs, generation_config, input_len

def compare_and_find_preferred_image(image_paths, processor, model):
    shuffled_image_dict = shuffle_image_dict(image_paths)
    prompt = construct_comparison_prompt(shuffled_image_dict)
    inputs, generation_config, input_len = prepare_inference(prompt, processor)
    outputs = model.generate(**inputs, generation_config=generation_config)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    content = processor.parse_response(response)["content"]
    preferred_image = int(content) - 1
    return shuffled_image_dict[preferred_image]['original_idx']

In [133]:
image_paths = ['./inputs/1.png', './inputs/2.png']
preferred = compare_and_find_preferred_image(image_paths, processor, model)
print(preferred)

577
1


In [115]:
messages = [
    {"role": "system", "content": "Your job is to decide which image you prefer."},
    {"role": "user", "content": [
        {"type": "text", "text": "Image A:"},
        {"type": "image", "path": "./inputs/1.png"},
        {"type": "text", "text": "Image B:"},
        {"type": "image", "path": "./inputs/2.png"},
        {"type": "text", "text": "Which image do you prefer? Only say A or B"},
    ]},
]

In [119]:
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=True,
).to(device)
input_len = inputs["input_ids"].shape[-1]
print(input_len)

571


In [120]:
generation_config = GenerationConfig(max_new_tokens=1024, early_stopping=True)
outputs = model.generate(**inputs, generation_config=generation_config)

In [121]:
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
print(response)

<|channel>thought
Thinking Process:

1.  **Analyze the Request:** The user has provided two images (Image A and Image B) and asked me to choose which one I prefer. The output must be strictly "A" or "B".

2.  **Analyze Image A:**
    *   **Subject:** A close-up profile of a person with light/silver hair.
    *   **Style:** Anime/Manga style.
    *   **Text/Context:** "The real fun for us has yet to come..."
    *   **Aesthetics:** Soft lighting, focus on facial expression/mood, elegant.

3.  **Analyze Image B:**
    *   **Subject:** A character with messy, spiky red hair, wearing glasses, and covered in red paint/blood across the face.
    *   **Style:** Anime/Manga style (intense/action-oriented).
    *   **Text/Context:** "I'll never forgive you! You'll see what I can do."
    *   **Aesthetics:** High intensity, aggressive, dramatic, focus on conflict/emotion.

4.  **Determine Preference (Subjective Choice):**
    *   Both images are visually strong within their respective genres (A 

In [83]:
output = processor.parse_response(response)
print(output)

{'role': 'assistant', 'thinking': 'Thinking Process:\n\n1.  **Analyze the Request:** The user has presented two images (Image A and Image B) and asked me to choose a preference (A or B).\n\n2.  **Analyze Image A:**\n    *   **Subject:** A muscular man (likely an anime character) with intense facial expressions (gritted teeth, determined/angry look).\n    *   **Style:** Action/Shonen anime style.\n    *   **Text/Context:** "If I say no?"\n    *   **Mood:** Intense, confrontational, serious.\n\n3.  **Analyze Image B:**\n    *   **Subject:** A woman with long, light/silver hair, looking off-camera.\n    *   **Style:** Anime/manga style.\n    *   **Text/Context:** "The real fun for us has yet to come..."\n    *   **Mood:** Calm, thoughtful, perhaps mysterious or hopeful.\n\n4.  **Determine Preference (Subjectivity Check):** Since there is no context or specific criteria (e.g., "I prefer action" or "I prefer beauty"), the choice is purely aesthetic or based on appeal.\n    *   Image A is dy